In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import DataLoader
from datasets import load_dataset
from collections import Counter

In [2]:
dataset = load_dataset(
    "Helsinki-NLP/opus_books",
    "en-fr"
)

print(dataset)
print("Training examples:", len(dataset["train"]))

README.md:   0%|          | 0.00/28.1k [00:00<?, ?B/s]

en-fr/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 21.0MB            

en-fr/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/127085 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['id', 'translation'],
        num_rows: 127085
    })
})
Training examples: 127085


In [3]:
vocab_size = 10_000

d_model = 32
num_heads = 4
num_layers = 2

context_length = 16

batch_size = 32

learning_rate = 0.001
num_epochs = 5

assert d_model % num_heads == 0

head_dim = d_model // num_heads

print("d_model:", d_model)
print("num_heads:", num_heads)
print("head_dim:", head_dim)
print("context_length:", context_length)
print("batch_size:", batch_size)

d_model: 32
num_heads: 4
head_dim: 8
context_length: 16
batch_size: 32


In [4]:
def tokenize(sentence):
    return sentence.lower().split()

In [5]:
english_counter = Counter()

for example in dataset["train"]:
    sentence = example["translation"]["en"]
    tokens = tokenize(sentence)
    english_counter.update(tokens)

print("Unique English words:", len(english_counter))

Unique English words: 137763


In [6]:
french_counter = Counter()

for example in dataset["train"]:
    sentence = example["translation"]["fr"]
    tokens = tokenize(sentence)
    french_counter.update(tokens)

print("Unique French words:", len(french_counter))

Unique French words: 160893


In [7]:
special_tokens = [
    "<PAD>",
    "<UNK>",
    "<BOS>",
    "<EOS>"
]

In [8]:
english_word_to_id = {
    "<PAD>": 0,
    "<UNK>": 1,
    "<BOS>": 2,
    "<EOS>": 3
}

for word, frequency in english_counter.most_common(vocab_size - 4):
    english_word_to_id[word] = len(english_word_to_id)

In [9]:
french_word_to_id = {
    "<PAD>": 0,
    "<UNK>": 1,
    "<BOS>": 2,
    "<EOS>": 3
}

for word, frequency in french_counter.most_common(vocab_size - 4):
    french_word_to_id[word] = len(french_word_to_id)

In [10]:
print("English vocabulary:", len(english_word_to_id))
print("French vocabulary:", len(french_word_to_id))

print(english_word_to_id["<PAD>"])
print(english_word_to_id["<BOS>"])

print(french_word_to_id["<PAD>"])
print(french_word_to_id["<BOS>"])

English vocabulary: 10000
French vocabulary: 10000
0
2
0
2


In [11]:
def encode(sentence, word_to_id):
    tokens = tokenize(sentence)

    ids = []

    for token in tokens:
        token_id = word_to_id.get(
            token,
            word_to_id["<UNK>"]
        )

        ids.append(token_id)

    ids = (
        [word_to_id["<BOS>"]]
        + ids
        + [word_to_id["<EOS>"]]
    )

    return ids

In [13]:
# padding

def pad_sequence(ids, max_length, pad_id):
    if len(ids) > max_length:
        ids = ids[:max_length]

    while len(ids) < max_length:
        ids.append(pad_id)

    return ids

In [14]:
# preprocess the entire dataset

english_data = []
french_data = []

for example in dataset["train"]:

    english_sentence = example["translation"]["en"]
    french_sentence = example["translation"]["fr"]

    english_ids = encode(
        english_sentence,
        english_word_to_id
    )

    french_ids = encode(
        french_sentence,
        french_word_to_id
    )

    english_ids = pad_sequence(
        english_ids,
        context_length,
        english_word_to_id["<PAD>"]
    )

    french_ids = pad_sequence(
        french_ids,
        context_length,
        french_word_to_id["<PAD>"]
    )

    english_data.append(english_ids)
    french_data.append(french_ids)

print("English examples:", len(english_data))
print("French examples:", len(french_data))

English examples: 127085
French examples: 127085


In [15]:
# convert the complete datasets into tensors

english_data = torch.tensor(
    english_data,
    dtype=torch.long
)

french_data = torch.tensor(
    french_data,
    dtype=torch.long
)

print("English data shape:", english_data.shape)
print("French data shape:", french_data.shape)

English data shape: torch.Size([127085, 16])
French data shape: torch.Size([127085, 16])


In [16]:
# create the DataLoader

train_loader = DataLoader(
    list(zip(english_data, french_data)),
    batch_size=batch_size,
    shuffle=True
)

print("Number of batches:", len(train_loader))

Number of batches: 3972


In [17]:
# create the embedding layers

english_token_embedding = nn.Embedding(
    len(english_word_to_id),
    d_model
)

french_token_embedding = nn.Embedding(
    len(french_word_to_id),
    d_model
)

position_embedding = nn.Embedding(
    context_length,
    d_model
)

In [19]:
d_model

32

In [20]:
# create the encoder Q/K/V projection weights

W_Q = nn.Linear(d_model, d_model)
W_K = nn.Linear(d_model, d_model)
W_V = nn.Linear(d_model, d_model)

W_O = nn.Linear(d_model, d_model)

In [21]:
# get a real batch from the full train_loader

src, tgt = next(iter(train_loader))

print("Source shape:", src.shape)
print("Target shape:", tgt.shape)

Source shape: torch.Size([32, 16])
Target shape: torch.Size([32, 16])


In [22]:
# Now turn the English IDs into the encoder input

src_vectors = english_token_embedding(src)

positions = torch.arange(context_length)

position_vectors = position_embedding(positions)

encoder_input = src_vectors + position_vectors

print("Encoder input shape:", encoder_input.shape)

Encoder input shape: torch.Size([32, 16, 32])


In [23]:
Q = W_Q(encoder_input)
K = W_K(encoder_input)
V = W_V(encoder_input)

print("Q shape:", Q.shape)
print("K shape:", K.shape)
print("V shape:", V.shape)

Q shape: torch.Size([32, 16, 32])
K shape: torch.Size([32, 16, 32])
V shape: torch.Size([32, 16, 32])


In [24]:
num_heads = 4
head_dim = 8

In [26]:
W_Q1 = nn.Linear(d_model, head_dim)
W_Q2 = nn.Linear(d_model, head_dim)
W_Q3 = nn.Linear(d_model, head_dim)
W_Q4 = nn.Linear(d_model, head_dim)

W_K1 = nn.Linear(d_model, head_dim)
W_K2 = nn.Linear(d_model, head_dim)
W_K3 = nn.Linear(d_model, head_dim)
W_K4 = nn.Linear(d_model, head_dim)

W_V1 = nn.Linear(d_model, head_dim)
W_V2 = nn.Linear(d_model, head_dim)
W_V3 = nn.Linear(d_model, head_dim)
W_V4 = nn.Linear(d_model, head_dim)

In [27]:
Q1 = W_Q1(encoder_input)
K1 = W_K1(encoder_input)
V1 = W_V1(encoder_input)

Q2 = W_Q2(encoder_input)
K2 = W_K2(encoder_input)
V2 = W_V2(encoder_input)

Q3 = W_Q3(encoder_input)
K3 = W_K3(encoder_input)
V3 = W_V3(encoder_input)

Q4 = W_Q4(encoder_input)
K4 = W_K4(encoder_input)
V4 = W_V4(encoder_input)

print("Head 1:", Q1.shape, K1.shape, V1.shape)
print("Head 2:", Q2.shape, K2.shape, V2.shape)
print("Head 3:", Q3.shape, K3.shape, V3.shape)
print("Head 4:", Q4.shape, K4.shape, V4.shape)

Head 1: torch.Size([32, 16, 8]) torch.Size([32, 16, 8]) torch.Size([32, 16, 8])
Head 2: torch.Size([32, 16, 8]) torch.Size([32, 16, 8]) torch.Size([32, 16, 8])
Head 3: torch.Size([32, 16, 8]) torch.Size([32, 16, 8]) torch.Size([32, 16, 8])
Head 4: torch.Size([32, 16, 8]) torch.Size([32, 16, 8]) torch.Size([32, 16, 8])


In [29]:
scores1 = Q1 @ K1.transpose(-2, -1)
scores2 = Q2 @ K2.transpose(-2, -1)
scores3 = Q3 @ K3.transpose(-2, -1)
scores4 = Q4 @ K4.transpose(-2, -1)

print("Head 1:", scores1.shape)
print("Head 2:", scores2.shape)
print("Head 3:", scores3.shape)
print("Head 4:", scores4.shape)

Head 1: torch.Size([32, 16, 16])
Head 2: torch.Size([32, 16, 16])
Head 3: torch.Size([32, 16, 16])
Head 4: torch.Size([32, 16, 16])


In [30]:
head_dim = 8

scores1 = scores1 / (head_dim ** 0.5)
scores2 = scores2 / (head_dim ** 0.5)
scores3 = scores3 / (head_dim ** 0.5)
scores4 = scores4 / (head_dim ** 0.5)

In [31]:
attention_weights1 = F.softmax(scores1, dim=-1)
attention_weights2 = F.softmax(scores2, dim=-1)
attention_weights3 = F.softmax(scores3, dim=-1)
attention_weights4 = F.softmax(scores4, dim=-1)

In [32]:
head1_output = attention_weights1 @ V1
head2_output = attention_weights2 @ V2
head3_output = attention_weights3 @ V3
head4_output = attention_weights4 @ V4

print("Head 1 output:", head1_output.shape)
print("Head 2 output:", head2_output.shape)
print("Head 3 output:", head3_output.shape)
print("Head 4 output:", head4_output.shape)

Head 1 output: torch.Size([32, 16, 8])
Head 2 output: torch.Size([32, 16, 8])
Head 3 output: torch.Size([32, 16, 8])
Head 4 output: torch.Size([32, 16, 8])


In [33]:
multi_head_output = torch.cat(
    [
        head1_output,
        head2_output,
        head3_output,
        head4_output
    ],
    dim=-1
)

print("Multi-head output shape:", multi_head_output.shape)

Multi-head output shape: torch.Size([32, 16, 32])


In [34]:
W_O = nn.Linear(
    d_model,
    d_model
)

In [35]:
attention_output = W_O(multi_head_output)

print("Attention output shape:", attention_output.shape)

Attention output shape: torch.Size([32, 16, 32])


In [36]:
residual_output = encoder_input + attention_output

print("Residual output shape:", residual_output.shape)

Residual output shape: torch.Size([32, 16, 32])


In [37]:
# Layer normalization

layer_norm = nn.LayerNorm(d_model)

normalized_output = layer_norm(residual_output)

print("Normalized output shape:", normalized_output.shape)

Normalized output shape: torch.Size([32, 16, 32])


In [38]:
ffn = nn.Sequential(
    nn.Linear(d_model, d_model * 4),
    nn.ReLU(),
    nn.Linear(d_model * 4, d_model)
)

In [39]:
ffn_output = ffn(normalized_output)

print("FFN output shape:", ffn_output.shape)

FFN output shape: torch.Size([32, 16, 32])


In [40]:
ffn_residual = normalized_output + ffn_output

print("FFN residual shape:", ffn_residual.shape)

FFN residual shape: torch.Size([32, 16, 32])


In [41]:
final_encoder_output = layer_norm(ffn_residual)

print("Final encoder output shape:", final_encoder_output.shape)

Final encoder output shape: torch.Size([32, 16, 32])
